## simple 1D spring based contact
two rigid bodies, target_rb, and end_effector_rb

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
#robot hardware
AF_FOIL_LENGTH = 0.8128 #32" blade to Meters

In [ ]:
import sympy as sp
import numpy as np
# --- Target Static Symbols ---
x_tar_sym, x_tar_dot_sym, x_tar_ddot_sym = sp.symbols(r'x_{targ} \dot{x}_{targ} \ddot{x}_{targ}')
y_tar_sym, y_tar_dot_sym, y_tar_ddot_sym = sp.symbols(r'y_{targ} \dot{y}_{targ} \ddot{y}_{targ}')
phi_tar_sym, phi_tar_dot_sym, phi_tar_ddot_sym = sp.symbols(r'\phi_{targ} \dot{\phi}_{targ} \ddot{\phi}_{targ}')

# --- End Effector Static Symbols ---
x_ee_sym, x_ee_dot_sym, x_ee_ddot_sym = sp.symbols(r'x_{ee} \dot{x}_{ee} \ddot{x}_{ee}')
y_ee_sym, y_ee_dot_sym, y_ee_ddot_sym = sp.symbols(r'y_{ee} \dot{y}_{ee} \ddot{y}_{ee}')
phi_ee_sym, phi_ee_dot_sym, phi_ee_ddot_sym = sp.symbols(r'\phi_{ee} \dot{\phi}_{ee} \ddot{\phi}_{ee}')

def getFullSubs(expression):
    # Substitute highest order derivatives FIRST!

    # --- TARGET SUBSTITUTIONS ---
    # x_tar
    expression = expression.subs(x_tar.diff().diff(), x_tar_ddot_sym)
    expression = expression.subs(x_tar.diff(), x_tar_dot_sym)
    expression = expression.subs(x_tar, x_tar_sym)

    # y_tar
    expression = expression.subs(y_tar.diff().diff(), y_tar_ddot_sym)
    expression = expression.subs(y_tar.diff(), y_tar_dot_sym)
    expression = expression.subs(y_tar, y_tar_sym)

    # phi_tar
    expression = expression.subs(phi_tar.diff().diff(), phi_tar_ddot_sym)
    expression = expression.subs(phi_tar.diff(), phi_tar_dot_sym)
    expression = expression.subs(phi_tar, phi_tar_sym)
    

    # --- END EFFECTOR SUBSTITUTIONS ---
    # x_ee
    expression = expression.subs(x_ee.diff().diff(), x_ee_ddot_sym)
    expression = expression.subs(x_ee.diff(), x_ee_dot_sym)
    expression = expression.subs(x_ee, x_ee_sym)

    # y_ee
    expression = expression.subs(y_ee.diff().diff(), y_ee_ddot_sym)
    expression = expression.subs(y_ee.diff(), y_ee_dot_sym)
    expression = expression.subs(y_ee, y_ee_sym)

    # phi_ee
    expression = expression.subs(phi_ee.diff().diff(), phi_ee_ddot_sym)
    expression = expression.subs(phi_ee.diff(), phi_ee_dot_sym)
    expression = expression.subs(phi_ee, phi_ee_sym)

    
    
    return expression
##Rigidbody dynamics
from sympy.physics.mechanics import dynamicsymbols
import sympy as sp
from IPython.display import display, Math
g, t = sp.symbols('g t')
#here mass is uniform, I may as well model with linear quadratic form for mass distribution
M_tar_rb, I_tar_rb_z= sp.symbols(r'M_{targ} I_{Z}^{targ}')
M_ee_rb, I_ee_rb_z= sp.symbols(r'M_ee I_Z^{ee}')

#NOTE: cool thing symetric matrix means energy is conserved, this is lagrangian so it is
# also we are only dealling with lumped-parameter modeling, not FEA meshes
#human_stiffness, blade_stiff = sp.symbols('k_targ k_ee')
#series_spring = human_stiffness + blade_stiff
k_tar_xx, k_tar_xy ,k_tar_yy, k_tar_phi = sp.symbols(r'^{targ}K_XX ^{targ}K_XY ^{targ}K_{YY} ^{targ}K_{\phi}')
k_blade_xx, k_blade_xy ,k_blade_yy, k_blade_phi = sp.symbols(r'^{foil}K_{XX} ^{foil}K_{XY} ^{foil}K_{YY}  ^{foil}K_{\phi}')

#k_tensor_tar = sp.Matrix([[k_tar_xx,k_tar_xy,0],[k_tar_xy,k_tar_yy,0],[0,0]])
#k_tensor_ee = sp.Matrix([[k_blade_xx,k_blade_xy],[k_blade_xy,k_blade_yy]])



m_tar_xx, m_tar_xy ,m_tar_yy = sp.symbols(r'M_{XX}^{targ} M_{XY}^{targ} M_{YY}^{targ}')
m_blade_xx, m_blade_xy ,m_blade_yy = sp.symbols(r'M_{XX}^{foil} M_{XY}^{foil} M_{YY}^{foil}')

mom_elasticity,blade_cross_section, blade_length,mom_2_area = sp.symbols('E_foil A_foil L_foil I_foil')

stiffness_matrix = sp.Matrix([[mom_elasticity*blade_cross_section/blade_length,0,0],
                             [0,(12*mom_elasticity*mom_2_area)/(blade_length**3),6*mom_elasticity*mom_2_area/blade_length**2],
                             [0,6*mom_elasticity*mom_2_area/blade_length**2, 4* mom_elasticity/(blade_length**3)]])

m_tensor_tar = sp.Matrix([[m_tar_xx,m_tar_xy,0],
                            [m_tar_xy,m_tar_yy,0],
                            [0,0,I_tar_rb_z]])

m_tensor_ee = sp.Matrix([[m_blade_xx,m_blade_xy,0],
                        [m_blade_xy,m_blade_yy,0],
                        [0,0,I_ee_rb_z]])


x_tar,y_tar,phi_tar= dynamicsymbols('x_targ y_targ phi_targ')

x_ee,y_ee,phi_ee= dynamicsymbols('x_ee y_ee phi_ee')

x_tar_dot = x_tar.diff()
y_tar_dot = y_tar.diff()
phi_tar_dot = phi_tar.diff()

x_tar_ddot = x_tar.diff().diff()
y_tar_ddot = y_tar.diff().diff()
phi_tar_ddot = phi_tar.diff().diff()

x_ee_dot = x_ee.diff()
y_ee_dot = y_ee.diff()
phi_ee_dot = phi_ee.diff()

x_ee_ddot = x_ee_dot.diff()
y_ee_ddot = y_ee_dot.diff()
phi_ee_ddot = phi_ee_dot.diff()


## robotic state
target_state_pos = sp.Matrix([x_tar,y_tar,phi_tar]);
end_effector_state_pos = sp.Matrix([x_ee,y_ee,phi_ee]);

target_state_vel = target_state_pos.diff()
end_effector_state_vel = end_effector_state_pos.diff();

display(getFullSubs(sp.Matrix.hstack(target_state_pos,end_effector_state_pos,target_state_vel,end_effector_state_vel)))




## lagrangian eqn setup, at collision

In [ ]:
#stiffness_series = k_tensor_tar #+ k_tensor_ee
resting_blade_length = np.array([AF_FOIL_LENGTH,0,0])
delta_spring_compress = (end_effector_state_pos - target_state_pos)+resting_blade_length

T =( 0.5  *(end_effector_state_vel.T*m_tensor_ee * end_effector_state_vel)[0] + .5  * (target_state_vel.T * m_tensor_tar*target_state_vel)[0] + .5 *(I_ee_rb_z*phi_ee + .5 *I_tar_rb_z*phi_tar))

V = .5 * (delta_spring_compress.T * stiffness_matrix  * delta_spring_compress )[0]
L = T-V
display(Math(
    sp.latex(getFullSubs(sp.Eq(sp.Symbol('L'),L)))
    ))



In [ ]:
#Damping
alpha_inertia_damping, beta_spring_stiffness_damping = sp.symbols(r'\alpha \beta')
C = alpha_inertia_damping * m_tensor_ee + beta_spring_stiffness_damping * stiffness_matrix
#source https://en.wikipedia.org/wiki/Rayleigh_dissipation_function
#dissapation Foil
R_dissapation_ee = .5 * end_effector_state_vel.T * C *end_effector_state_vel
display(Math("R=" + sp.latex(getFullSubs(R_dissapation_ee[0]))))

In [ ]:
from IPython.display import Math
q = sp.Matrix([end_effector_state_pos,target_state_pos])
L_dQ_sys_1 = sp.Matrix([L.diff(q_state) for q_state in q ])

L_dQ_dot_sys_1 = sp.Matrix([L.diff(q_state.diff()) for q_state in q ])

L_dQ_dot_dT_sys_1 = sp.Matrix([L.diff(q_state.diff()).diff(t) for q_state in q ])

rayleigh = sp.Matrix([R_dissapation_ee.diff(q_state.diff()) for q_state in q])
eqn1 = (L_dQ_dot_dT_sys_1-L_dQ_sys_1) + rayleigh

display(Math(
    sp.latex(
        getFullSubs(eqn1)
        )+ "=" + sp.latex(sp.Matrix(q))
    ))


In [ ]:
#collision eqns
# Group your target acceleration variables into a tuple/list
accel_vars = (x_ee_ddot, y_ee_ddot, phi_ee_ddot, x_tar_ddot, y_tar_ddot, phi_tar_ddot)

# This forces it to untangle the M_XY couplings!
solved_accels = sp.solve(eqn1, accel_vars)

# solved_accels will return a dictionary mapping each variable to its explicit solution.
explicit_accel_matrix = sp.Matrix([
    solved_accels[x_ee_ddot],
    solved_accels[y_ee_ddot],
    solved_accels[phi_ee_ddot],
    solved_accels[x_tar_ddot],
    solved_accels[y_tar_ddot],
    solved_accels[phi_tar_ddot]
])

# Display the final, decoupled equations of motion
display(Math( sp.latex(getFullSubs(explicit_accel_matrix)) + "="  + sp.latex(getFullSubs(sp.Matrix(q.diff().diff())))))
#display(Math(
#    sp.latex(getFullSubs( sp.Matrix([col_eq_x_ee,col_eq_y_ee,col_eq_phi_ee,col_eq_x_targ,col_eq_y_targ,col_eq_phi_targ]))) + "="  + sp.latex(getFullSubs(sp.Matrix(q.diff().diff())))
#))


## Lambdify sympy to numpy for scipy

In [ ]:
distance_down_from_blade_tip = np.array([0,0,0]) 
physical_constants = {

# --- Blade Geometric & Material Properties (Maraging Steel) ---

mom_elasticity: 190e9, # E: Young's Modulus of steel (~200 GPa)

blade_length: AF_FOIL_LENGTH -distance_down_from_blade_tip[0], # L: Standard length of a foil blade (0.90 meters)NOTE:this is actually the length of contact from base

blade_cross_section: 1e-5, # A: Cross-sectional area (approx. 15 mm^2)

mom_2_area: 1.2e-11, # I: Area moment of inertia (highly dependent on cross-section shape)


# --- End Effector (Foil) Mass Properties ---

# Total mass of a foil is ~0.5 kg.

m_blade_xx: 0.5, # Mass resisting X-translation

m_blade_yy: 0.5, # Mass resisting Y-translation

m_blade_xy: 0.0, # Off-diagonal mass coupling (0 if symmetric center of mass)

I_ee_rb_z: 0.135, # Rotational inertia (approx. 1/3 * m * L^2 for a rod pivoted at base)


# --- Target Mass Properties ---

# Assuming a heavy dummy or the torso of an opponent (~50 kg effective mass)

m_tar_xx: 50.0, # Target mass resisting X-translation

m_tar_yy: 50.0, # Target mass resisting Y-translation

m_tar_xy: 0.0, # Target mass coupling

I_tar_rb_z: 2.5 , # Target rotational inertia


alpha_inertia_damping:0.5,

beta_spring_stiffness_damping:.02,


} 

# 1. Dynamically create plain symbols for positions and velocities based on your 'q' list.
# (This assumes 't' is your SymPy time symbol, e.g., t = sp.Symbol('t'))
q_sym = [sp.Symbol(var.func.__name__) for var in q]
q_dot_sym = [sp.Symbol(var.func.__name__ + '_d') for var in q]

# 2. Build the substitution dictionary to wipe out derivatives and time-dependence
calc_to_alg_map = {}
for func_var, sym_var, sym_dot_var in zip(q, q_sym, q_dot_sym):
    calc_to_alg_map[func_var] = sym_var
    calc_to_alg_map[sp.diff(func_var, t)] = sym_dot_var

# 3. Create the state_vars tuple containing ONLY plain symbols (Positions first, then Velocities)
# NOTE: When you call these functions in your ODE solver, you must pass the arguments 
# in this exact order: (x_ee, y_ee, phi_ee, x_tar, y_tar, phi_tar, x_ee_d, y_ee_d, ...)
state_vars = tuple(q_sym + q_dot_sym)

# 4. Clean the expressions and Lambdify
# ---------------------------------------------------------
# END EFFECTOR ACCELERATIONS
# ---------------------------------------------------------
compute_accel_x_ee = sp.lambdify(
    state_vars, 
    solved_accels[x_ee_ddot].subs(physical_constants).subs(calc_to_alg_map), 
    modules="numpy"
)

compute_accel_y_ee = sp.lambdify(
    state_vars, 
    solved_accels[y_ee_ddot].subs(physical_constants).subs(calc_to_alg_map), 
    modules="numpy"
)

compute_accel_phi_ee = sp.lambdify(
    state_vars, 
    solved_accels[phi_ee_ddot].subs(physical_constants).subs(calc_to_alg_map), 
    modules="numpy"
)

# ---------------------------------------------------------
# TARGET ACCELERATIONS
# ---------------------------------------------------------
compute_accel_x_tar = sp.lambdify(
    state_vars, 
    solved_accels[x_tar_ddot].subs(physical_constants).subs(calc_to_alg_map), 
    modules="numpy"
)

compute_accel_y_tar = sp.lambdify(
    state_vars, 
    solved_accels[y_tar_ddot].subs(physical_constants).subs(calc_to_alg_map), 
    modules="numpy"
)

compute_accel_phi_tar = sp.lambdify(
    state_vars, 
    solved_accels[phi_tar_ddot].subs(physical_constants).subs(calc_to_alg_map), 
    modules="numpy"
)

In [ ]:
# ──────────────────────────────────────────────────────────
# Blade deflection boundary limits
# ──────────────────────────────────────────────────────────
Y_BOUND   = 0.1    # max blade tip deflection in Y (±0.1 m)
K_WALL    = 1e5    # penalty spring stiffness (N/m)
DAMP_WALL = 1e2    # penalty damping (N·s/m)
M_BLADE   = 0.5    # blade mass (kg) — must match physical_constants
M_TARG    = 50.0   # target mass (kg) — must match physical_constants

def blade_target_collide(t, state):
    # Unpack the 12-dimensional state vector
    x_ee_sim, x_ee_d, y_ee_sim, y_ee_d, phi_ee_sim, phi_ee_d, \
    x_targ, x_targ_d, y_targ, y_targ_d, phi_targ, phi_targd = state

    # Pass ALL 12 variables (6 positions + 6 velocities) to every compute function!
    # The order must match state_vars exactly: positions first, then velocities.
    args = (x_ee_sim, y_ee_sim, phi_ee_sim, x_targ, y_targ, phi_targ,
            x_ee_d, y_ee_d, phi_ee_d, x_targ_d, y_targ_d, phi_targd)

    x_ee_dd     = compute_accel_x_ee(*args)
    y_ee_dd     = compute_accel_y_ee(*args)
    phi_ee_dd   = compute_accel_phi_ee(*args)
    x_targ_dd   = compute_accel_x_tar(*args)
    y_targ_dd   = compute_accel_y_tar(*args)
    phi_targ_dd = compute_accel_phi_tar(*args)

    # ── Y-axis boundary clamping ──────────────────────────
    # If the blade tip deflects beyond ±Y_BOUND, a stiff penalty
    # spring pushes it back and transfers the reaction force to
    # the target's X-axis (axial push, Option A).
    if abs(y_ee_sim) > Y_BOUND:
        penetration = y_ee_sim - (Y_BOUND if y_ee_sim > 0 else -Y_BOUND)
        wall_force  = -K_WALL * penetration - DAMP_WALL * y_ee_d

        # Apply wall force to blade Y (pushes tip back inside boundary)
        y_ee_dd += wall_force / M_BLADE

        # Transfer reaction to target X-axis (blade is rigid → pushes target)
        x_targ_dd += (-wall_force) / M_TARG

    # Return the derivatives [velocity, acceleration, ...]
    return [
        x_ee_d, x_ee_dd,
        y_ee_d, y_ee_dd,
        phi_ee_d, phi_ee_dd,
        x_targ_d, x_targ_dd,
        y_targ_d, y_targ_dd,
        phi_targd, phi_targ_dd
    ]

def blade_contact_lost(t_val, state):
    x_ee_sim, x_ee_d, y_ee_sim, y_ee_d, phi_ee_sim, phi_ee_d, \
    x_targ, x_targ_d, y_targ, y_targ_d, phi_targ, phi_targd = state

    point_1 = np.array([x_ee_sim, y_ee_sim])
    polor_coordiate_convert = AF_FOIL_LENGTH * np.array([np.cos(phi_ee_sim), np.sin(phi_ee_sim)])
    point_2 = point_1 + polor_coordiate_convert
    rb = np.array([x_targ, y_targ])

    mag_back_blade  = np.linalg.norm(point_1 - rb)
    mag_front_blade = np.linalg.norm(point_2 - rb)

    thickness_tolerance = 0.05
    return (mag_back_blade + mag_front_blade) - (AF_FOIL_LENGTH + thickness_tolerance)


In [ ]:
# Define realistic physical constants for a fencing foil and target
# Units are standard SI: kg, meters, radians, seconds, Pascals (N/m^2)
distance_down_from_blade_tip = np.array([0,0,0]) #angle is last val
physical_constants = {
    # --- Blade Geometric & Material Properties (Maraging Steel) ---
    mom_elasticity: 190e9,       # E: Young's Modulus of steel (~200 GPa)
    blade_length: AF_FOIL_LENGTH -distance_down_from_blade_tip[0],          # L: Standard length of a foil blade (0.90 meters)NOTE:this is actually the length of contact from base
    blade_cross_section: 1e-5,  # A: Cross-sectional area (approx. 15 mm^2)
    mom_2_area: 1.2e-11,         # I: Area moment of inertia (highly dependent on cross-section shape)

    # --- End Effector (Foil) Mass Properties ---
    # Total mass of a foil is ~0.5 kg. 
    m_blade_xx: 0.5,             # Mass resisting X-translation
    m_blade_yy: 0.5,             # Mass resisting Y-translation
    m_blade_xy: 0.0,             # Off-diagonal mass coupling (0 if symmetric center of mass)
    I_ee_rb_z: 0.135,            # Rotational inertia (approx. 1/3 * m * L^2 for a rod pivoted at base)

    # --- Target Mass Properties ---
    # Assuming a heavy dummy or the torso of an opponent (~50 kg effective mass)
    m_tar_xx: 50.0,              # Target mass resisting X-translation
    m_tar_yy: 50.0,              # Target mass resisting Y-translation
    m_tar_xy: 0.0,               # Target mass coupling 
    I_tar_rb_z: 2.5  ,            # Target rotational inertia

    alpha_inertia_damping:0.5,
    beta_spring_stiffness_damping:.02,

    
}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

blade_contact_lost.terminal = True
blade_contact_lost.direction = 1

t_span = (0.0, 2.0)
t_eval = np.linspace(t_span[0], t_span[1], 5000)
#t_eval = np.linspace(t_span[0], t_span[1], 1000) # 1000 frames of data

init_boundry_cond = [
    0, -20,  # x_ee, x_ee_vel
    0 ,0.01,  # y_ee, y_ee_vel
    0, 0,  # phi_ee, phi_ee_vel
    physical_constants[blade_length], 2,  # x_targ (starts 0.5m away), x_targ_vel
    distance_down_from_blade_tip[1], 0.0,  # y_targ, y_targ_vel
    distance_down_from_blade_tip[2], 0.0   # phi_targ, phi_targ_vel
]

y0 = init_boundry_cond
sol_function = []


sol = solve_ivp(blade_target_collide, t_span, y0, t_eval=t_eval,events=blade_contact_lost,method="Radau",rtol=1e-3,atol=1e-3)
sol_function.append(sol)



In [ ]:
import random
import os
import pandas as pd
import time
import multiprocessing as mp
from multiprocessing import cpu_count

# ──────────────────────────────────────────────────────────
# Worker function: runs a single simulation
# ──────────────────────────────────────────────────────────
def run_single_sim(sim_config):
    """Run one ODE simulation with the given initial conditions.
    Returns (sim_id, DataFrame) on success, or (sim_id, None) on failure.
    """
    sim_id, x_ee_vel, x_targ_vel, y_ee_vel = sim_config


    t_span = (0.0, 2.0)
    t_eval = np.linspace(t_span[0], t_span[1], 5000)

    y0 = [
        0, x_ee_vel,                                          # x_ee, vx_ee
        0, y_ee_vel,                                              # y_ee, vy_ee
        0, 0,                                                 # phi_ee, vphi_ee
        physical_constants[blade_length], x_targ_vel,         # x_targ, vx_targ
        distance_down_from_blade_tip[1], 0.0,                 # y_targ, vy_targ
        distance_down_from_blade_tip[2], 0.0                  # phi_targ, vphi_targ
    ]

    sol = solve_ivp(blade_target_collide, t_span, y0,
                    t_eval=t_eval, events=blade_contact_lost,
                    method="Radau", rtol=1e-3, atol=1e-3)

    if not sol.success:
        return sim_id, None

    dt_array = np.diff(sol.t)

    data = {
        'sim_id': sim_id,
        't': sol.t[:-1],
        'dt': dt_array,

        # --- INPUT CONSTANTS ---
        'c_E': physical_constants[mom_elasticity],
        'c_L': physical_constants[blade_length],
        'c_A': physical_constants[blade_cross_section],
        'c_I': physical_constants[mom_2_area],
        'c_m_blade': physical_constants[m_blade_xx],
        'c_I_blade': physical_constants[I_ee_rb_z],
        'c_m_targ': physical_constants[m_tar_xx],
        'c_I_targ': physical_constants[I_tar_rb_z],
        'c_alpha_damp': physical_constants[alpha_inertia_damping],
        'c_beta_damp': physical_constants[beta_spring_stiffness_damping],

        # --- INPUT DATA (State at time t) ---
        'in_x_ee': sol.y[0][:-1],
        'in_vx_ee': sol.y[1][:-1],
        'in_y_ee': sol.y[2][:-1],
        'in_vy_ee': sol.y[3][:-1],
        'in_phi_ee': sol.y[4][:-1],
        'in_vphi_ee': sol.y[5][:-1],
        'in_x_targ': sol.y[6][:-1],
        'in_vx_targ': sol.y[7][:-1],
        'in_y_targ': sol.y[8][:-1],
        'in_vy_targ': sol.y[9][:-1],
        'in_phi_targ': sol.y[10][:-1],
        'in_vphi_targ': sol.y[11][:-1],

        # --- OUTPUT DATA (Label/State at time t + dt) ---
        'out_x_ee': sol.y[0][1:],
        'out_vx_ee': sol.y[1][1:],
        'out_y_ee': sol.y[2][1:],
        'out_vy_ee': sol.y[3][1:],
        'out_phi_ee': sol.y[4][1:],
        'out_vphi_ee': sol.y[5][1:],
        'out_x_targ': sol.y[6][1:],
        'out_vx_targ': sol.y[7][1:],
        'out_y_targ': sol.y[8][1:],
        'out_vy_targ': sol.y[9][1:],
        'out_phi_targ': sol.y[10][1:],
        'out_vphi_targ': sol.y[11][1:]
    }

    return sim_id, pd.DataFrame(data)


# ──────────────────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────────────────
dataset = 100
n_workers = max(1, cpu_count() - 1)
print(f"Using {n_workers} parallel workers ({cpu_count()} CPU cores available)")

blade_contact_lost.terminal = True
blade_contact_lost.direction = 1

# Use 'fork' context explicitly — Jupyter may default to 'forkserver'
# which cannot pickle functions defined in notebook cells.
fork_ctx = mp.get_context('fork')

# ──────────────────────────────────────────────────────────
# Training Data (100 sims in parallel)
# ──────────────────────────────────────────────────────────
print(f"\nGenerating {dataset} training simulations...")
t_start = time.time()

train_configs = []
for i in range(1, dataset + 1):
    x_ee_vel = random.uniform(5.0, 25.0)
    x_targ_vel = random.uniform(-5.0, 5.0)
    y_ee_vel = random.uniform(-5.0, 5.0)    # blade tip Y velocity (bidirectional)
    train_configs.append((i, x_ee_vel, x_targ_vel, y_ee_vel))

with fork_ctx.Pool(processes=n_workers) as pool:
    train_results = pool.map(run_single_sim, train_configs)

train_frames = []
for sim_id, df in train_results:
    if df is not None:
        train_frames.append(df)
        display((sim_id, "success"))
    else:
        print(f"Sim {sim_id} failed.")

t_train = time.time() - t_start
print(f"Training sims completed in {t_train:.1f}s ({t_train/dataset:.2f}s/sim avg)")

# Save training data (append to existing file if present)
new_data = pd.concat(train_frames, ignore_index=True)
if os.path.exists("master_training_dataset.csv"):
    existing_data = pd.read_csv("master_training_dataset.csv")
    master_dataset = pd.concat([existing_data, new_data], ignore_index=True)
    print(f"Appended {len(new_data)} rows to existing {len(existing_data)} rows.")
else:
    master_dataset = new_data
master_dataset.to_csv("master_training_dataset.csv", index=False)
print(f"Finished — master_training_dataset.csv now has {len(master_dataset)} total rows.")

# ──────────────────────────────────────────────────────────
# Validation/Label Data (20 sims in parallel)
# ──────────────────────────────────────────────────────────
n_val = int(dataset * 0.2)
print(f"\nGenerating {n_val} validation simulations...")
t_start = time.time()

sim_id_offset = dataset
val_configs = []
for i in range(1, n_val + 1):
    x_ee_vel = random.uniform(5.0, 25.0)
    x_targ_vel = random.uniform(-5.0, 5.0)
    y_ee_vel = random.uniform(-5.0, 5.0)    # blade tip Y velocity (bidirectional)
    val_configs.append((sim_id_offset + i, x_ee_vel, x_targ_vel, y_ee_vel))

with fork_ctx.Pool(processes=n_workers) as pool:
    val_results = pool.map(run_single_sim, val_configs)

val_frames = []
for sim_id, df in val_results:
    if df is not None:
        val_frames.append(df)
        display((sim_id, "success"))
    else:
        print(f"Sim {sim_id} failed.")

t_val = time.time() - t_start
print(f"Validation sims completed in {t_val:.1f}s ({t_val/n_val:.2f}s/sim avg)")

# Save validation data (append to existing file if present)
new_data = pd.concat(val_frames, ignore_index=True)
if os.path.exists("master_label_dataset.csv"):
    existing_data = pd.read_csv("master_label_dataset.csv")
    master_dataset = pd.concat([existing_data, new_data], ignore_index=True)
    print(f"Appended {len(new_data)} rows to existing {len(existing_data)} rows.")
else:
    master_dataset = new_data
master_dataset.to_csv("master_label_dataset.csv", index=False)
print(f"Finished — master_label_dataset.csv now has {len(master_dataset)} total rows.")

print(f"\nTotal time: {t_train + t_val:.1f}s")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Load the most recent data
df = pd.read_csv("master_training_dataset.csv")

# Pick one simulation to visualise
sim_ids = df['sim_id'].unique()
chosen_sim = sim_ids[0]  # change this to inspect different sims
sim_data = df[df['sim_id'] == chosen_sim]

print(f"Visualising sim_id = {chosen_sim}  ({len(sim_data)} samples)")

# Compute accelerations (force / mass) from finite differences
dt = sim_data['dt'].values
accel_names = {
    'x_ee':   ('in_vx_ee',   'out_vx_ee'),
    'y_ee':   ('in_vy_ee',   'out_vy_ee'),
    'phi_ee': ('in_vphi_ee', 'out_vphi_ee'),
    'x_targ': ('in_vx_targ', 'out_vx_targ'),
    'y_targ': ('in_vy_targ', 'out_vy_targ'),
    'phi_targ': ('in_vphi_targ', 'out_vphi_targ'),
}

t = sim_data['t'].values

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
fig.suptitle(f'Accelerations (≈ Forces) — Sim {chosen_sim}', fontsize=14)

for ax, (name, (v_in_col, v_out_col)) in zip(axes.flatten(), accel_names.items()):
    v_in  = sim_data[v_in_col].values
    v_out = sim_data[v_out_col].values
    accel = (v_out - v_in) / dt

    # Color positive (red) and negative (blue) forces
    pos = accel.copy(); pos[pos < 0] = 0
    neg = accel.copy(); neg[neg > 0] = 0

    ax.fill_between(t, pos, 0, color='red', alpha=0.4, label='Positive')
    ax.fill_between(t, neg, 0, color='blue', alpha=0.4, label='Negative')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f'ä_{name}')
    ax.set_ylabel('m/s²')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1, 0].set_xlabel('Time (s)')
axes[-1, 1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

# Summary: % positive vs negative per DOF
print("\n--- Force direction summary ---")
for name, (v_in_col, v_out_col) in accel_names.items():
    v_in  = sim_data[v_in_col].values
    v_out = sim_data[v_out_col].values
    accel = (v_out - v_in) / dt
    pct_pos = (accel > 0).mean() * 100
    pct_neg = (accel < 0).mean() * 100
    print(f"  {name:10s}: {pct_neg:.0f}% negative, {pct_pos:.0f}% positive")

In [ ]:
import matplotlib.pyplot as plt

# Extract the solution object from your list
sol = sol_function[0]

if sol.success:
    t = sol.t
    
    # Extract End Effector (Foil) positions
    x_ee = sol.y[0]
    y_ee = sol.y[2]
    phi_ee = sol.y[4]
    
    # Extract Target positions
    x_targ = sol.y[6]
    y_targ = sol.y[8]
    phi_targ = sol.y[10]
    
    # Create a 3-panel figure
    fig, axs = plt.subplots(3, 1, figsize=(10, 12), sharex=True)
    
    # --- Plot 1: X-Axis (Axial Compression) ---
    axs[0].plot(t, x_ee, label='Foil $X_{ee}$', color='blue', linewidth=2)
    axs[0].plot(t, x_targ, label='Target $X_{targ}$', color='red', linestyle='--', linewidth=2)
    axs[0].set_title('Axial Dynamics (X-axis)')
    axs[0].set_ylabel('Position (m)')
    axs[0].grid(True, linestyle=':')
    axs[0].legend(loc='upper right')
    
    # --- Plot 2: Y-Axis (Transverse Bending) ---
    axs[1].plot(t, y_ee, label='Foil $Y_{ee}$', color='blue', linewidth=2)
    axs[1].plot(t, y_targ, label='Target $Y_{targ}$', color='red', linestyle='--', linewidth=2)
    axs[1].set_title('Transverse Bending Dynamics (Y-axis)')
    axs[1].set_ylabel('Position (m)')
    axs[1].grid(True, linestyle=':')
    axs[1].legend(loc='upper right')
    
    # --- Plot 3: Phi (Rotation) ---
    axs[2].plot(t, phi_ee, label=r'Foil $\phi_{ee}$', color='blue', linewidth=2)
    axs[2].plot(t, phi_targ, label=r'Target $\phi_{targ}$', color='red', linestyle='--', linewidth=2)
    axs[2].set_title(r'Rotational Dynamics ($\phi$)')
    axs[2].set_xlabel('Time (s)')
    axs[2].set_ylabel('Angle (rad)')
    axs[2].grid(True, linestyle=':')
    axs[2].legend(loc='upper right')
    
    # Adjust layout to prevent overlap and display
    plt.tight_layout()
    plt.show()

else:
    print(f"Simulation failed to converge: {sol.message}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import Video, display

# Extract the solution object
sol = sol_function[0]

if sol.success:
    t = sol.t
    # Node 1 (Base/Guard)
    x_ee, y_ee, phi_ee = sol.y[0], sol.y[2], sol.y[4]
    
    # Node 2 (Tip/Target Contact Point)
    x_targ, y_targ, phi_targ = sol.y[6], sol.y[8], sol.y[10]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Dynamic scaling based on the full simulation envelope
    padding = 0.1
    ax.set_xlim(min(np.min(x_ee), np.min(x_targ)) - padding, 
                max(np.max(x_ee), np.max(x_targ)) + padding)
    ax.set_ylim(min(np.min(y_ee), np.min(y_targ)) - padding, 
                max(np.max(y_ee), np.max(y_targ)) + padding)
    
    ax.set_aspect('equal')
    ax.grid(True, linestyle=':')
    ax.set_title('FEA Beam Deflection (Cubic Hermite Interpolation)')
    
    # --- VISUALIZATION ELEMENTS ---
    # FEA Dots representing the nodes along the beam
    dots_blade, = ax.plot([], [], 'b.', markersize=.1, label='FEA Interpolated Nodes', zorder=2)
    line_blade, = ax.plot([], [], 'b-', lw=1.5, alpha=0.6, zorder=1) # Faint line connecting dots
    
    # Tip Trajectory Trace
    trace_tip, = ax.plot([], [], 'r--', lw=5.0, alpha=0.5, label='Tip Trajectory', zorder=0)
    
    # Rigid body markers for the endpoints
    dot_base, = ax.plot([], [], 'ko', markersize=1, label='Node 1 (Base)')
    dot_targ, = ax.plot([], [], 'ro', markersize=1, label='Node 2 (Tip/Target)')
    
    ax.legend(loc='upper right')

    # Define the number of elements/dots to calculate along the blade
    num_dots = 20
    # xi represents the normalized local coordinate (0 at base, 1 at tip)
    xi = np.linspace(0, 1, num_dots)

    def init():
        dots_blade.set_data([], [])
        line_blade.set_data([], [])
        dot_base.set_data([], [])
        dot_targ.set_data([], [])
        trace_tip.set_data([], [])
        return dots_blade, line_blade, dot_base, dot_targ, trace_tip

    def update(frame):
        # Get current global states for Node 1 and Node 2
        x1, y1, phi1 = x_ee[frame], y_ee[frame], phi_ee[frame]
        x2, y2, phi2 = x_targ[frame], y_targ[frame], phi_targ[frame]
        
        # 1. Calculate Chord Kinematics (The straight line between nodes)
        dx = x2 - x1
        dy = y2 - y1
        Lc = np.sqrt(dx**2 + dy**2) # Dynamic chord length (captures axial strain)
        alpha = np.arctan2(dy, dx)  # Global chord angle
        
        # 2. Calculate Local Node Rotations (Relative to the chord)
        theta1 = phi1 - alpha
        theta2 = phi2 - alpha
        
        # 3. Apply Euler-Bernoulli Cubic Hermite Shape Functions
        # N2 and N4 map end-rotations to internal transverse deflections
        N2 = xi * (1 - xi)**2
        N4 = xi**2 * (xi - 1)
        
        # Local kinematics (u is axial stretching, v is transverse bending)
        u_local = xi * Lc
        v_local = Lc * (N2 * theta1 + N4 * theta2)
        
        # 4. Transform Local Frame back to Global Coordinates
        x_blade = x1 + u_local * np.cos(alpha) - v_local * np.sin(alpha)
        y_blade = y1 + u_local * np.sin(alpha) + v_local * np.cos(alpha)
        
        # Update visuals
        dots_blade.set_data(x_blade, y_blade)
        line_blade.set_data(x_blade, y_blade)
        dot_base.set_data([x1], [y1])
        dot_targ.set_data([x2], [y2])
        
        # Update the historical trace up to the current frame
        trace_tip.set_data(x_targ[:frame+1], y_targ[:frame+1])
        
        return dots_blade, line_blade, dot_base, dot_targ, trace_tip

    # GPU Render Setup
    skip_frames = max(1, len(t) // 400) 
    frames_to_render = np.arange(0, len(t), skip_frames)
    anim = FuncAnimation(fig, update, frames=frames_to_render, init_func=init, blit=True)

    video_filename = "fea_deflection_gpu.mp4"
    # Note: h264_nvenc for NVIDIA. Use h264_amf for AMD, h264_videotoolbox for Mac.
    writer = FFMpegWriter(fps=50, bitrate=2500, extra_args=['-vcodec', 'h264_nvenc', '-preset', 'fast'])

    print("Interpolating FEA nodes and encoding on GPU...")
    anim.save(video_filename, writer=writer)
    plt.close(fig)
    
    display(Video(video_filename, embed=True, html_attributes="controls autoplay loop"))

else:
    print("Cannot animate: Simulation failed.")